In [ ]:
##########################################
#                                        #
#                D(E)W(E)I               #
#       IDK - Something DWI related      #
#                                        #
##########################################
"v5.yes Dewey from malcolm in the middle, MA"

### Version 02.02.2026 @ MA Pyhton3.12 -- Linux version 
# based on last dwi seq.
# does not use BIDS derivatives structure, but assumes BIDS format for
# reading in the raw data sss
# If you prj directory doesn't match BIDS structure, please consider running 'BIDS_v2.py' first
#
# To DO before 'official release' 
# work a bit more on T1w integration 
# fix fsleyes - supervised mode bug
# add log file
# Set up steps counting to not hard code folders number

###############################################################
###############################################################
#
# Dewei expects folder structure like:
# Whatever BIDS....
#        dwi
#            |Prep
#            |Denoise 
#            |Gibbs 
#            etc...
# Each step has its own folder.
# Si nun te piace, amen stacce
###############################################################
###############################################################

#Home made utilities - These paths lead to bash and python functions


In [1]:
"""
SET UP lib. and main flags
---------------------------------------------
DEBUGGING / LOGGING FLAGS
"""
bash_func=r"/home/malberti/wks14/temp/FF_DWI_Drift/script/bash_func"  #path to bash script  add, cwd=bash_func 
python_func = "/home/malberti/wks14/temp/FF_DWI_Drift/script/python_func"

import subprocess
import os
import getpass
from datetime import datetime
import sys
import socket
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, PillowWriter
import pandas as pd
from time import time
import nibabel as nib
import glob as gl
#import nilearn as nil
import math
import time

## DIPY 
# nifti manipualtion 
from dipy.io.image import load_nifti, save_nifti
from scipy.ndimage import binary_dilation
from dipy.core.gradients import gradient_table
from dipy.io.gradients import read_bvals_bvecs

from dipy.sims.voxel import multi_tensor
from dipy.core.sphere import disperse_charges, HemiSphere
from dipy.data import get_fnames, small_sphere
from dipy.reconst.csdeconv import auto_response_ssst

# Model fitting 
import dipy.reconst.fwdti as fwdti
import dipy.reconst.dki as dki
import dipy.reconst.msdki as msdki ## I've never implemented this model because there are not enough info from litterature 
from dipy.reconst.dki_micro import axonal_water_fraction
import dipy.reconst.dki_micro as dki_micro
from dipy.reconst.ivim import IvimModel

sys.path.append(python_func)

#from EDDY_RMS import do_EDDY_RMS, do_RMS_movement_plots
from Personalized_mask import do_personalized_mask
#from  RMS_movement_plots import do_RMS_movement_plots
from Supervised_mode import do_supervised_mode
from Tensor_plot import do_tensor_plot
from HomeMadeSlicesDir import do_HomeMadeSlicesDir 


## AMICO (Noddi) -- Settings are based on SK's noddi code
#import amico
#amico.setup()
#ae = amico.Evaluation()
#ae.set_config('doComputeNRMSE', True) #'doDebiasSignal', True)              #'doSaveModulatedMaps', True)

subprocess.run(f'clear')
do_log = 1  # Print log file after each step. Checks if the output is there.
do_debug=1 #enable supervised mode and open each output on fsleyes

/home/malberti/Unix_Folders/SWEEP2/DEWEY_v7-01/dewey_v7-01/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
##Log file
def LOGsOut_update(LOGs_dir, step, dwi_output=None, sub_id=None, ses_id=None):  #LOGsOut, Step, subjid  --> better it is here atm sub_id=subjid, ses_id=session)
    """
    Open a TXT file, located in the path folder. 
    If the file doesn't exist, it create it - the file is subject/session specific

    Each step will write down what is going on, the commands, errors etcc... 
    futhermore, this Log file report: 

    - Time of day/Day 
    - username and IP 
    - Last step 
    - Command History 
    - mean value of each volume - [output of mrstat]
    - parameters es. f and g for bet
    """ 
    subjid_ss=f"{sub_id}_{ses_id}"
    LOGs=os.path.join(LOGs_dir, f"{subjid_ss}-LOGs.txt")
    if not os.path.exists(LOGs_dir):
        os.makedirs(LOGs_dir)
    if step == "Setting up files":
        with open(LOGs, "w") as f:
            f.write("#########################\n")
            f.write("## DEWEY LOGs FILE !!! ##\n")
            f.write("#########################\n\n\n\n")
            f.write(f"{getpass.getuser()}\n")
            f.write(f"{socket.gethostname()}\n")
            f.write(f"{socket.gethostbyname(socket.gethostname())}\n")
            f.write(f"{datetime.now()}\n")
            f.write(f"Subject BIDS ID: {subjid_ss}\n\n\n")
            
            f.write(f"Selected preprocessing step & settings:\n")
            f.write(f"Usuall: 0='no/skip' & 1='yes/let's do it' \n")
            f.write(f"do_prep: {do_prep}\n")
            f.write(f"do_denoise: {do_denoise}\n")
            f.write(f"do_gibbsring: {do_gibbsring}\n")
            f.write(f"do_topup: {do_topup}\n")
            f.write(f"do_eddy: {do_eddy}\n")
            f.write(f"do_bias: {do_bias}\n")
            f.write(f"do_dtifit: {do_dtifit}\n")
            f.write(f"do_kurtosis: {do_kurtosis}\n")
            f.write(f"do_noddi: {do_noddi}\n\n\n")

            f.write(f"do_personalized_mask: {do_mask}\n")
            f.write(f"do_debug: {do_debug}\n")
            f.write("#########################\n\n\n\n\n")
    else:
        with open(LOGs, "a") as f:
            
            f.write(f"  Ref.step: {step}\n")
            f.write(f"{getpass.getuser()}\n")
            f.write(f"{socket.gethostbyname(socket.gethostname())}\n")
            f.write(f"{datetime.now()}\n")
            f.write(f"To visualize the last output type: [fsleyes {dwi_output}]\n")
            f.write(f"Below the mean value of each volume of the last output, please controll that it is different from zero\n")
            
            if "rad" in dwi_output:
                    ## We are checking the radiant conversion before running denoising: 
                f.write(f"The values muse be comprises between -3.14 and +3.14, if not stop dewei and call me")
                """ To get a better estimation, you may apply a mask to this image"""
                subprocess.run(f"mrstats {dwi_output} -ignorezero >> {LOGs}",shell=True) 
            else: 
                subprocess.run(f"mrstats {dwi_output} -ignorezero >> {LOGs}",shell=True) 

## SELECT STEPS ##

In [4]:
do_prep=1 # HIGHLY RECOMEND and study specific, otherwise the script doesn't (almost) work 

# Preprocessing
do_denoise=0 #Run MP-PCA denoising, mandatory if number of volumes is higher that 32
do_gibbsring=0 # you can run it by using a stand alone script after index estimation
do_topup=1 # Just run it and shut up
do_eddy=1 # Same with potatoes 
do_bias=0  #run ANTs N4biascorrection, almost useless (?) - not yeat implemented
do_signaldrift=1 #Run patch wise signal drift correction

# Scalar maps
do_dtifit=1  # ---> Dtifit
do_kurtosis=0  
do_noddi=0
do_freewater=0

# Utilities
do_coreg=0
do_smoothing=0
do_tractography=0 # DEWEI can perform a whole brain tractography using msmt_csd algorithm. 

# NODDI  params
# The model is setted on: 
# dPar ==> intrinsic parallel diffusivity; 0.0017 for WM but 0.0011 for GM! aka. Parallel diffusivity [mm^2/s].
# dIso ==> Maybe should higher in GM, aka  Isotropic diffusivity [mm^2/s]. 
# IC_VFs ==> Intra-cellular volume fractions. (default is np.linspace(0.1, 0.99, 12))
# IC_ODs ==> Intra-cellular orientation dispersions. (default is np.hstack((np.array([0.03, 0.06]), np.linspace(0.09, 0.99, 10))))

dPar = 0.0011  
dIso = 0.003

## MASKING & DENOISE setting ##
# if it sets to 1, every time that the script creates a mask, a while loop will start. 
# it allows the used to check bet mask and/or change bet f and g value to get the best mask as possible 
# Use this flag with asian (just a bit of racism sorry) or small brain

do_mask=1 ## FSL BET or dwi2mask? The script uses bet, but you can replace if with dwi2mask if you need something quicker 
f=0.4  # base/standard f and g values 
g=0

phase_avail=1 # If phase image is available, please set it to 1 and the script will run denoising on complex data instead Mag. file only
phase_range=4096 #it is usually estimate by ((max+(-min))/2. Phase must be in radiants and the range is [-pi..pi] 

#smoothing 
l2fwhm=[1.69865806, 2.54798709]   # KERNEL 4 mni space smoothing in mm l2fwhm=(4 6)


## STANDARD PATH SETTING 

In [5]:
## Set Path & Steps settings
config=r"/usr/local/fsl/etc/flirtsch/b02b0.cnf"  # Topup config file

### set path for fsl and freesurfer
fsl_path=r"/usr/local/fsl"

### set path to reference images 
# 1mm 
refT11mm=r"/usr/local/fsl/data/standard/MNI152_T1_1mm.nii.gz"                             # MNI 1mm image 
refT11mmBrain=r"/usr/local/fsl/data/standard/MNI152_T1_1mm_brain.nii.gz"              # MNI 1mm brain extraction

# refT11mmBrainMask = "$fsl_path/data/standard/MNI152_T1_1mm_brain_mask_dil.nii.gz"   # for T1 and T2, _dil to be conservative with final brain mask
refT11mmBrainMask=r"/usr/local/fsl/data/standard/MNI152_T1_1mm_brain_mask.nii.gz"        # for T1 and T2, _dil to be conservative with final brain mask
refT11mmBrainMaskTight=r"/usr/local/fsl/data/standard/MNI152_T1_1mm_brain_mask.nii.gz"

# 2mm 
refT12mm=r"/usr/local/fsl/data/standard/MNI152_T1_2mm.nii.gz"                         # MNI 2mm T1 image 
refT12mmBrain=r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz"                 # MNI 2mm brain extraction
refT12mmBrainMask=r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain_mask_dil.nii.gz"     # for T1 and T2, _dil to be conservative with final brain mask

### path to ANTs templates - Usefull only fot T1w preprocessing
ants_template_path=r"/home/malberti/Unix_Folders/MRI_templates/OASIS30ANTs"
ants_template=r"tpl-OASIS30ANTs_res-01_desc-brain_T1w.nii.gz"
ants_prior=r"tpl-OASIS30ANTs_res-01_label-brain_probseg.nii.gz" 

## SET WORKING DIRECTORY
prj_path=r"/home/malberti/wks14/temp/FF_DWI_Drift" # main BIDS path /home/malberti/Unix_Folders/Sweep/Pilot/SW005
project="sub-" #Common name for folders and files, for example all my sweep files are named as "sw001_ss1_dwi_AP.nii || sw001_ss1_T1w.nii"
script=r'/home/malberti/wks14/temp/FF_DWI_Drift/Script'

## Sessions and participants set up

In [6]:
vps = [45]
#vps = [15,16,17]
METRICS = {
    "Tensor":   ["FA", "MD"],
    "Kurtosis": ["MK", "kMD", "kFA"],
    "NODDI":    ["NDI", "ODI", "FWF"],
    "FreeWater":["fw", "fwMD", "fwFA"]
} #These are the metric that the script can estimate - In future i will add more 

In [7]:
#Acquparams: 
index=r'/home/malberti/wks14/temp/FF_DWI_Drift/Script/index.txt'

## Preparation

In [8]:
"""
The first block 'Setting up files...' it is study specific and hard coded, you need to rearrange/change it based on you data set... The output, must be a seq that looks like: 
PA --> AP --> PA 
+ bvecs and bvals files
"""

NdoMinchiaSiamo="Setting up files" #Define the step on the log file
#subprocess.run(f" rm -rf LOGs", shell=True) #Remove old logs 

n_threads = 7
processes = []

if do_prep == 1:
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
        raw_sub_path = os.path.join(prj_path, "rawdata", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
    
        for session in sessions: 
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            orig = os.path.join(prj_path, "rawdata", id, session, "dwi")
            derivatives = os.path.join(prj_path, "derivatives", id, session)

            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")  # Log directory
            LOGsOut_update(LOGsOut,NdoMinchiaSiamo,dwi_output=None,sub_id=id, ses_id=session)
            LOGs_file=os.path.join(LOGsOut, f"{subjid}-LOGs.txt")

            print("⚙️ Starting DWI prep... (merging/splitting/checking images)")
            # Check if derivatives folder exists
            if not os.path.isdir(derivatives):
                os.makedirs(os.path.join(derivatives, "dwi", "prep"))
            else:
                # Check if dwi_prep folder exists
                if not os.path.isdir(dwi_prep):
                    os.makedirs(os.path.join(dwi_prep, "prep"))
                else:
                    # Check if dwi_prep/prep exists
                    prep_path = os.path.join(dwi_prep, "prep")
                    if not os.path.isdir(prep_path):
                        os.makedirs(prep_path)
            
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path

            """
            The script expect 2 PA images and 2 AP, respectively phase and magnitude. AP and PA have the same number of volumes/bval, we can easily merge everything 
            """
            
            DwIs=[]
            BVALs=[]
            BVECs=[]
            
            # --- Wait if we've hit the parallel job limit ---
            while len(processes) >= n_threads:
                time.sleep(1)  # Avoid busy-waiting / CPU spinning
                processes = [p for p in processes if p.poll() is None]  # Keep only running ones

            # --- Launch subprocess ---
            parpool = subprocess.Popen(f"./DEWEY_prep.sh -i {orig} -out {dwi_prep} -subjid {subjid} -log {LOGs_file}",cwd=bash_func,shell=True)
            processes.append(parpool)
            print(f" Active jobs: {len(processes)}/{n_threads}")

    for p in processes:
        p.wait()

⚙️ Starting DWI prep... (merging/splitting/checking images)
 Active jobs: 1/7
⚙️ Starting DWI prep... (merging/splitting/checking images)
 DWI preparation (AP + PA merge)
Original data path : /home/malberti/wks14/temp/FF_DWI_Drift/rawdata/sub-45/ses-01/dwi
Output path        : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep
Data type          : 
Subject ID         : sub-45_ses-01

Merging AP + PA (mag) → /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii
 Active jobs: 2/7
⚙️ Starting DWI prep... (merging/splitting/checking images)
 DWI preparation (AP + PA merge)
Original data path : /home/malberti/wks14/temp/FF_DWI_Drift/rawdata/sub-45/ses-02/dwi
Output path        : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep
Data type          : 
Subject ID         : sub-45_ses-02

Merging AP + PA (mag) → /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep/sub

## Write acqparams files

In [9]:
import json
num_b0s = 6 # Bos * AP/PA
num_b0s = 6  # b0s per direction (AP or PA)

for vp in vps:
    id = f"sub-{vp:02d}"

    raw_sub_path = os.path.join(prj_path, "rawdata", id)
    sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
    
    for session in sessions:

        subjid = f"{id}_{session}"
        LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")
        LOGs_file = os.path.join(LOGsOut, f"{subjid}-LOGs.txt")

        derivatives = os.path.join(prj_path, "derivatives", id, session)
        topup_path = os.path.join(derivatives, "dwi", "topup")

        orig = os.path.join(prj_path, "rawdata", id, session, "dwi")
        json_file = os.path.join(orig, f"{subjid}_dir-AP_part-mag_dwi.json")

        acqparams = os.path.join(topup_path, f"{subjid}_acqparams.txt")

        os.makedirs(topup_path, exist_ok=True)

        # -----------------------------
        # Read dwell time from JSON
        # -----------------------------
        with open(json_file, "r") as f:
            js = json.load(f)

        dwell = js["TotalReadoutTime"]

        # -----------------------------
        # Build acqparams lines
        # -----------------------------
        lines = []

        for _ in range(num_b0s):
            lines.append(f"0 1 0 {dwell}")

        for _ in range(num_b0s):
            lines.append(f"0 -1 0 {dwell}")

        # -----------------------------
        # Save file
        # -----------------------------
        with open(acqparams, "w") as f:
            f.write("\n".join(lines) + "\n")

        print(f"Saved {acqparams}")

Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/sub-45_ses-01_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/sub-45_ses-02_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-03/dwi/topup/sub-45_ses-03_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-04/dwi/topup/sub-45_ses-04_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-05/dwi/topup/sub-45_ses-05_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-06/dwi/topup/sub-45_ses-06_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-07/dwi/topup/sub-45_ses-07_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-08/dwi/topup/sub-45_ses-08_acqparams.txt
Saved /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-09/dwi/topup/sub-45_ses-09_acqparams.txt
Saved /home/malbert

##  DENOSING -Running MP-PC

print("\n\n")
print("###############################\n")
print("##  DENOSING -Running MP-PCA ##\n")
print("###############################\n")
print("\n\n")

If you are running it without preparation you should change the .sh file 

In [ ]:
if do_denoise == 1:
    
    NdoMinchiaSiamo="Denoising"

    """
    The denoising is performed by using MP-PCA algorithm on complex data. 
    DEWEI will run the denosing on each, individually AP block in order to avoid issues and mistakes given by
    motion artifacts due to saliva sampling and blocks
    we will run denoising on: 
    1st PA + 1st AP 
    2ndAP 
    3trAP + 4th PA 
    in the end, the script merge everything together and prepare the sequence to run topup correction on denoised B0s

    Phase in radiant and complex data will be obtained by using mrcalc, it apply generic voxel-wise mathematical operations to images
    USAGE
    mrcalc [ options ] operand [ operand ... ]

    operand an input image, intensity value, or the special keywords
            'rand' (random number between 0 and 1) or 'randn' (random
            number from unit std.dev. normal distribution) or the
            mathematical constants 'e' and 'pi'.

    DESCRIPTION
    This command will only compute per-voxel operations. Use 'mrmath' to compute summary statistics across images or along image axes.
    This command uses a stack-based syntax, with operators (specified using options) operating on the top-most entries (i.e. images or values) in the
    stack. Operands (values or images) are pushed onto the stack in the order they appear (as arguments) on the command-line, and operators (specified as options) operate on and consume the top-most entries in the stack, and push their output as a new entry on the stack.
    As an additional feature, this command will allow images with different dimensions to be processed, provided they satisfy the following conditions: for each axis, the dimensions match if they are the same size, or one of them has size one. In the latter case, the entire image will be
    replicated along that axis. This allows for example a 4D image of size [ X Y Z N ] to be added to a 3D image of size [ X Y Z ], as if it consisted of N copies of the 3D image along the 4th axis (the missing dimension is assumed to have size 1). Another example would a single-voxel 4D image of
    size [ 1 1 1 N ], multiplied by a 3D image of size [ X Y Z ], which would allow the creation of a 4D image where each volume consists of the 3D image scaled by the corresponding value for that volume in the single-voxel image.
    """

    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)

        raw_sub_path = os.path.join(prj_path, "rawdata", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
    
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 

            derivatives = os.path.join(prj_path, "derivatives", id, session)   # Path to derivative folder for this participant & session
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            
            LOGsOut = os.path.join(script, "LOGs", f"{subjid}-LOGs")  # Log directory
            tt_log=os.path.join(script, "LOGs", f"{subjid}-LOGs",f"{subjid}-LOGs.txt" )

                # Set denoising path
            denoise_path=os.path.join(derivatives, "dwi","denoise")
            if not os.path.exists(denoise_path):
                os.makedirs(denoise_path)
            
            print("⚙️ Starting DWI denoising...")
            print(f"\nProcessing subj: {subjid}.....") # must be here and JustToBeSure

            """
            The denoise bash script takes as input the common filename pattern
                ({subjid}_dir-PA-AP_part)
            which is shared across magnitude, phase, and complex images.

            The script also requires:
                - an input folder
                - an output folder

            All other naming conventions and processing steps are fully automatic and hard-coded in the script.

            Output files (all saved in {denoise_path}):

            1) Radiant phase image
            ]]{subjid}_dir-PA-AP_part-rad_dwi.nii.gz
            Phase image converted to radians.

            2) Complex image
            {subjid}_dir-PA-AP_part-complex_dwi.nii.gz
            Complex-valued image obtained by combining magnitude and radiant phase.

            3) Denoised magnitude image
            {subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz
            Denoised DWI using MPPCA.

            4) Noise estimate
            {subjid}_dir-PA-AP_part-mag_dwi_noise.nii.gz
            Noise kernel estimated via mean MPPCA.

            5) Residual image
            {subjid}_dir-PA-AP_part-mag_dwi_residue.nii.gz
            Difference between the raw and denoised magnitude images.

            To add or remove specific flags in the dwidenoise call,
            please edit the DEWEY_denoise script.
            """

            DwIs = f"{subjid}_dir-PA-AP" # DWImri input - already PA/AP merged 

            if not os.path.exists(os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_C.nii.gz")): #This line is specific for SWEEP2, feel free to remove it 
                    subprocess.run(f"./DEWEY_denoise_v1-02.sh -i {DwIs} -in_path {dwi_prep} -phase {phase_avail} -out_path {denoise_path} -log {tt_log}", cwd=bash_func, shell=True) #Run denoising_mppca.sh -noise {noise_DwIS}
                
            if do_debug ==1:
                DwIs = os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")  # Define input DWI magnitude file
                out_DwIs=os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz")  # Define output denoised file
                noise_DwIS=os.path.join(denoise_path, f"{subjid}_dir-PA-AP_noise.nii.gz")   # Define noise map file produced by denoising
                residue_DwIs=os.path.join(denoise_path, f"{subjid}_dir-PA-AP_residue.nii.gz")   # Define residuals file (original - denoised)
                do_supervised_mode(DwIs,out_DwIs,noise_DwIS,residue_DwIs) #open fsleyes 
            
            LOGsOut_update(LOGsOut, NdoMinchiaSiamo, out_DwIs, subjid)

##  Running GIBBSring

print("\n\n")
print("#############################\n")
print("##  Running GIBBSring ...  ##\n")
print("#############################\n")
print("\n\n")

In [ ]:
if do_gibbsring == 1:
    NdoMinchiaSiamo="Gibbs ring"

    """
    ----------------------------------------
    Gibbs ringing removal step
    This step applies MRtrix3's degibbsing using sub-voxel shifts
    MRtrix3's mrdegibbs algorithm reduces Gibbs artifacts by estimating
    the local signal along each axis and applying a **sub-voxel shift**
    to minimize oscillations near edges without blurring the image - at least not too much
    ----------------------------------------
    """
    
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
        project_id = f"sub-2{vp:02d}"  # project_id: project-specific code (sub-2XX) — yes, a bit redundant
        
        raw_sub_path = os.path.join(prj_path, "rawdata", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
    
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            LOGsOut = os.path.join(script, "LOGs", f"{subjid}-LOGs")  # Log directory

        dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
        in_DwIs_prep=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz") 
        
        denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
        in_DwIs_denoised=os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz") 

        gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
        residue_DwIs=os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_residue.nii.gz")   

        if not os.path.exists(gibbs_path):
            os.makedirs(gibbs_path)  

        # ---------------------------
        # Run degibbsing
        # ---------------------------

        if os.path.exists(in_DwIs_denoised): #The script select which input dwi - if exists the script will always choose the denoised one
            in_DwIs=in_DwIs_denoised
            out_DwIs=os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs.nii.gz")
            subprocess.run(f"./DEWEY_degibbs_v1-02.sh -i {in_DwIs} -res {residue_DwIs} -out {out_DwIs}",cwd=bash_func, shell=True)
        else:
            in_DwIs=in_DwIs_prep
            out_DwIs=os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_degibbs.nii.gz")
            subprocess.run(f"./DEWEY_degibbs_v1-02.sh  -i {in_DwIs} -res {residue_DwIs} -out {out_DwIs}",cwd=bash_func, shell=True)
        
        if do_debug ==1:
            do_supervised_mode(in_DwIs,out_DwIs,residue_DwIs)
        
        # do_HomeMadeSlicesDir requires as input bvecs and bvals because it create a slicedir image*bvalue and in order to sort/move across bvals it uses these two files
        
        grad1=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bval") # add $set.bvec/bval
        grad2=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bvec")
        output_png=f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs.png"
        do_HomeMadeSlicesDir(residue_DwIs, grad1, grad2, gibbs_path, output_png) 
        LOGsOut_update(LOGsOut, NdoMinchiaSiamo, out_DwIs, subjid)  # Update logs

##  TOPUP finally

print("\n\n")
print("##########################################################\n")
print("##  TOPUP finally, have you checked acqparams file?...  ##\n")
print("##########################################################\n")
print("\n\n")

In [10]:
if do_topup == 1:
    NdoMinchiaSiamo="TOPUP"
    config= r"/usr/local/fsl/etc/flirtsch/b02b0.cnf" #FSL config file

    """
    Topup Correction (FSL)

    --- Description ---
    Topup is used to correct for **susceptibility-induced distortions** in diffusion MRI.
    It estimates the spatial field inhomogeneity (in Hz) caused by magnetic susceptibility,
    then applies a correction to the images. Typically, it is applied to the b0 volumes
    of DWI data acquired with opposite phase-encoding directions (e.g., PA and AP).

    --- Compulsory arguments ---
    You MUST provide at least one of the following:
        --imain      : 4D file containing the images to correct (usually extracted b0s)

    --- Optional arguments ---
        --acqp       : Text file specifying phase-encoding directions and readout times
        --out        : Base name for output spline coefficients (Hz) and movement parameters
        --fout       : Name of image file with estimated field (Hz)
        --iout       : Name of 4D image file with unwarped images
        --featout    : Base name for exporting results to FEAT
        --nthr       : Number of threads to use (default = 1; cannot exceed hardware cores)
        -h, --help   : Display help information
        -v, --verbose: Print diagnostic information during processing

    --- How to run Topup ---
    1. Extract all b0 images from your main DWI dataset.
    2. Set up the acquisition parameters in a text file. Example format:
        0  1  0  [total readout time]   # for PA
        0 -1  0  [total readout time]   # for AP
    3. Run topup using the extracted b0s and the acquisition parameters.

    --- References ---
    More info: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/EDDY/Faq#Topup
    """
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)   
        raw_sub_path = os.path.join(prj_path, "derivatives", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])

        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")  # Log directory
            LOGs_file=os.path.join(script, "LOGs", f"{id}-LOGs",f"{subjid}-LOGs.txt" )
            derivatives = os.path.join(prj_path, "derivatives", id, session)   # Path to derivative folder for this participant & session

            topup_path=os.path.join(derivatives, "dwi","topup")
            topup_out=os.path.join(topup_path, f"topup_out")  #topup out common name
            acqparams = os.path.join(topup_path, f"{subjid}_acqparams.txt")
            
            if not os.path.exists(topup_path):
                os.makedirs(topup_path)
            
            """ 
            To preprocess the sweep data, i need to change index and acquparams file: 
            A --> ses-01 and ses-04
            B --> ses-02 and ses-05
            C --> ses-03 and ses-06
            

            A = ["ses-01", "ses-04"]
            B = ["ses-02", "ses-05"]
            C = ["ses-03", "ses-06"]

            if session in ["ses-01", "ses-04"]:
                suffix = "A"
            elif session in ["ses-02", "ses-05"]:
                suffix = "B"
            elif session in ["ses-03", "ses-06"]:
                suffix = "C"

            acqparams = f"/home/malberti/wks14/temp/FF_DWI_Drift/script/acqparams_{suffix}.txt"
            index = f"/home/malberti/wks14/temp/FF_DWI_Drift/script/index_{suffix}.txt"
            """
            # Determine the appropriate DWI output to use:
            # The script automatically selects the DWI file in the following priority:
            #   1. Degibbsed (Gibbs ringing corrected)
            #   2. Denoised
            #   3. Raw (original) DWI
            # Note: The script assumes that preprocessing steps are performed sequentially.
            #      es. If the Gibbs ringing correction flag is set (do_gibbsring = 1) but
            #       the expected output does not exist, the script will exit to avoid errors.

            dwi_prep = os.path.join(derivatives,"dwi", "prep")  # Step path
            in_DwIs_prep=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii") 

            """  Set up SWEEP2 preprocessing analysis - comment these lines if u are processing other data than sub-981 to sub-999 
            src_file = os.path.join(derivatives, f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")
            print(src_file)
            if os.path.exists(src_file):
                os.makedirs(dwi_prep, exist_ok=True)
                subprocess.run(f"mv {os.path.join(derivatives,subjid+'_dir-PA-AP_part-mag_dwi.nii.gz')} {dwi_prep}/.",shell=True)
                subprocess.run(f"mv {os.path.join(derivatives,subjid+'_dir-PA-AP_part-mag_dwi.bvec')} {dwi_prep}/.",shell=True)
                subprocess.run(f"mv {os.path.join(derivatives, subjid+'_dir-PA-AP_part-mag_dwi.bval')} {dwi_prep}/.",shell=True)
                """
            
        
            in_DwIs = in_DwIs_prep
           
            # Topup is estimated on b0s, which are extracted directly within the bach command.
            b0s=os.path.join(topup_path, f"{subjid}_dir-PA-AP_part-mag_dwi_b0s.nii.gz") #This file will contain all the b0s and it will be the input for topup
            fslgrad=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi") # add $set.bvec/bval the script needs it to extract b0s

            #if not os.path.exists(os.path.join(topup_path, f"topup_out_unwarped.nii.gz")):
             # --- Wait if we've hit the parallel job limit ---
            while len(processes) >= n_threads:
                time.sleep(1)
                processes = [p for p in processes if p.poll() is None]

            # --- Launch subprocess ---
            parpool = subprocess.Popen(f"./DEWEY_topup_v1-02.sh -i {in_DwIs} -b {b0s} -fslgrad {fslgrad} -acq {acqparams} -log {LOGs_file} -out {topup_out}", cwd=bash_func,shell=True)
            processes.append(parpool)
            print(f"Active jobs: {len(processes)}/{n_threads}")

    # --- Wait for ALL remaining jobs to finish ---
    for p in processes:
        p.wait()

                
    """  
        # ---------------------------
        # Define Topup outputs for inspection and logging
        # ---------------------------
        topup_out_field = os.path.join(topup_path, "topup_out_rfields.nii.gz")   # Estimated field (Hz)
        topup_out_warp  = os.path.join(topup_path, "topup_out_unwarped.nii.gz")  # Unwarped DWI
        topup_rms       = os.path.join(topup_path, "topup_out_movpar.txt")        # Movement parameters
        topup_rms_out   = os.path.join(topup_path, "topup_out_")                  # Prefix for RMS plots - the RMS after topup are computed on b0s only

        # ---------------------------
        # QC & visualization
        # ---------------------------
        # Generate RMS movement plots to inspect head motion
        do_RMS_movement_plots(topup_rms, topup_rms_out)

        if do_debug==1:
            # Open supervised inspection on fsleyes for b0s, unwarped images, and estimated field
            do_supervised_mode(b0s, topup_out_warp, topup_out_field)

        # Update log to record the Topup step
        LOGsOut_update(LOGsOut,NdoMinchiaSiamo,dwi_output=topup_out_warp,sub_id=id, ses_id=session)
    """

Active jobs: 7/7
 Running DEWEY_topup.sh
Input DWI          : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii
B0 images          : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/sub-45_ses-01_dir-PA-AP_part-mag_dwi_b0s.nii.gz
Gradient files     :  (.bvec/.bval)
Acquisition params : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/sub-45_ses-01_acqparams.txt
Topup output prefix: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/topup_out
Topup R-Field      : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/topup_out_rfields
Topup Unwarped     : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/topup/topup_out_unwarped

 Extracting B0 volumes...


dwiextract: [WARNING] existing output files will be overwritten
dwiextract: [DEBUG] No config file found at "/etc/mrtrix.conf"
dwiextract: [DEBUG] No config file found at "/home/malberti/.mrtrix.conf"
dwiextract: [INFO] opening image "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii"...
dwiextract: [DEBUG] memory-mapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii"...
dwiextract: [DEBUG] file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii" mapped at 0x70c778600000, size 243936352 (read-only)
dwiextract: [DEBUG] transforms_match: FOV difference in scanner coordinates: 0
dwiextract: [DEBUG] unmapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.nii"
dwiextract: [DEBUG] sanitising image information...
dwiextra

Active jobs: 2/7
Active jobs: 3/7
Active jobs: 4/7
 Running DEWEY_topup.sh
Input DWI          : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep/sub-45_ses-02_dir-PA-AP_part-mag_dwi.nii
B0 images          : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/sub-45_ses-02_dir-PA-AP_part-mag_dwi_b0s.nii.gz
Gradient files     :  (.bvec/.bval)
Acquisition params : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/sub-45_ses-02_acqparams.txt
Topup output prefix: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/topup_out
Topup R-Field      : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/topup_out_rfields
Topup Unwarped     : /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/topup/topup_out_unwarped

 Extracting B0 volumes...
Active jobs: 5/7
Active jobs: 6/7
Active jobs: 7/7
 Running DEWEY_topup.sh
Input DWI          : /home/malberti/wks14/temp/F

dwiextract: [DEBUG] memory-mapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-03/dwi/prep/sub-45_ses-03_dir-PA-AP_part-mag_dwi.nii"...
dwiextract: [DEBUG] memory-mapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-04/dwi/prep/sub-45_ses-04_dir-PA-AP_part-mag_dwi.nii"...
dwiextract: [DEBUG] found 3x72 matrix in file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep/sub-45_ses-02_dir-PA-AP_part-mag_dwi.bvec"
dwiextract: [DEBUG] b-value scaling: max scaling factor = exp(1.1298403615995964e-06) = 1.0000011298409999
dwiextract: [INFO] found 72x4 diffusion gradient table
dwiextract: [DEBUG] searching for suitable phase encoding data...
dwiextract: [DEBUG] memory-mapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep/sub-45_ses-02_dir-PA-AP_part-mag_dwi.nii"...
dwiextract: [DEBUG] file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-02/dwi/prep/sub-45_ses-02_dir-PA-AP_part

 Running Topup...


...

 Running Topup...


======]
====]
dwiextract: [DEBUG] image "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-13/dwi/topup/sub-45_ses-13_dir-PA-AP_part-mag_dwi_b0s.nii.gz" unloaded
dwiextract: [DEBUG] unmapping file "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-13/dwi/prep/sub-45_ses-13_dir-PA-AP_part-mag_dwi.nii"
dwiextract: [DEBUG] image "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-13/dwi/prep/sub-45_ses-13_dir-PA-AP_part-mag_dwi.nii" unloaded
mrstats: uncompressing image "/home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-13/dwi/topup/sub-45_ses-13_dir-PA-AP_part-mag_dwi_b0s.nii.gz"... [===

 Running Topup...


===============================================]


 Running Topup...
SSD = 260.747	n = 96195	Reg = 0	Cost = 260.747
SSD = 261.214	n = 96195	Reg = 0	Cost = 261.214
SSD = 260.463	n = 96195	Reg = 0	Cost = 260.463
SSD = 262.203	n = 96195	Reg = 0	Cost = 262.203
SSD = 262.954	n = 96195	Reg = 0	Cost = 262.954
SSD = 173.643	n = 96195	Reg = 0	Cost = 173.643
SSD = 120.515	n = 96195	Reg = 7.09325	Cost = 127.608
SSD = 120.893	n = 96195	Reg = 7.12912	Cost = 128.022
SSD = 120.217	n = 96195	Reg = 7.06975	Cost = 127.287
SSD = 121.178	n = 96195	Reg = 7.08764	Cost = 128.265
SSD = 121.253	n = 96195	Reg = 7.09114	Cost = 128.344
SSD = 88.6484	n = 96195	Reg = 5.22627	Cost = 93.8747
SSD = 68.4867	n = 96195	Reg = 15.6243	Cost = 84.111
SSD = 68.6851	n = 96195	Reg = 15.6976	Cost = 84.3827
SSD = 68.3353	n = 96195	Reg = 15.5771	Cost = 83.9124
SSD = 68.9449	n = 96195	Reg = 15.6742	Cost = 84.6191
SSD = 68.707	n = 96195	Reg = 15.6417	Cost = 84.3487
SSD = 61.0071	n = 96195	Reg = 12.1768	Cost = 73.1839
SSD = 53.9929	n = 96195	Reg = 20.3091	Cost = 74.302
SSD = 54.0832	

##  EDDY

print("\n\n")
print("##########################################\n")
print("##  EDDY finally - mo si bestemmia ...  ##\n")
print("##########################################\n")
print("\n\n")

In [11]:
n_threads=5
if do_eddy == 1:
    NdoMinchiaSiamo="EDDY"    

    """
        This section of the script handles motion and eddy current correction:

        1. Runs FSL's eddy current correction on the DWI data to correct for:
        - Eddy currents induced by diffusion gradients
        - Subject motion during the scan

        2. Creates a brain mask from the Topup unwarped output:

        3. Optionally runs FSL's MCFLIRT in parallel to:
        - Quantify participant movement before and after correction
        - Generate plots of motion parameters for quality control

        4. Provides an automatic, self-supervised mask option:
        - Uses either FSL BET or MRtrix's dwi2mask or ANTs --> select if from 
        - Selected based on user preference
        - Must be configured before running this step

        5. Generates an Eddy-QUAD report for automated quality assessment:
        - Summarizes motion, eddy residuals, and overall data quality
        - Useful for identifying problematic scans or subjects

        Notes:
        - Ensure the mask generation method and eddy parameters are properly set before execution.
        - This step assumes that Topup has been completed and outputs are available.
    """
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)   
        raw_sub_path = os.path.join(prj_path, "derivatives", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
   
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")  # Log directory
            LOGs_file=os.path.join(script, "LOGs", f"{id}-LOGs",f"{subjid}-LOGs.txt" )
            derivatives = os.path.join(prj_path, "derivatives", id, session)   # Path to derivative folder for this participant & session

            ## Path setting
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            in_DwIs_prep=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii") 
            
            denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
            in_DwIs_denoised=os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz") 

            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            # Possible degibbsed files: denoised+degibbs and raw+degibbs
            in_DwIs_degibbs = [
                os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs.nii.gz"),
                os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_degibbs.nii.gz")
            ]

            # Check which degibbsed files exist
            existing_degibbs = [f for f in in_DwIs_degibbs if os.path.exists(f)]

            if len(existing_degibbs) == 1:   # Exactly one degibbsed file exists → use it
                in_DwIs = existing_degibbs[0]
            elif len(existing_degibbs) > 1:  # Both degibbsed files exist - use the denoised one
                in_DwIs=in_DwIs_degibbs[0]
            else:       # No degibbsed files exist → check denoised output
                if os.path.exists(in_DwIs_denoised):
                    in_DwIs = in_DwIs_denoised
                elif os.path.exists(in_DwIs_prep):
                    in_DwIs = in_DwIs_prep
                else:
                    # Nothing exists → exit with error
                    print(f"Error: No input file found for {subjid}.")
                    print(f"Checked degibbsed files, denoised output, and raw DWI.")
                    exit(1)
            
            topup_path=os.path.join(derivatives, "dwi","topup")
            topup_out=os.path.join(topup_path, f"topup_out")  #topup out common name
            acqparams = os.path.join(topup_path, f"{subjid}_acqparams.txt")
            
            eddy_path=os.path.join(derivatives, "dwi","eddy")
            if not os.path.exists(eddy_path):
                os.makedirs(eddy_path)

            topup_out=os.path.join(topup_path, f"topup_out")  #topup out common name
            fslgrad=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi") # add $set.bvec/bval

            eddy_out=os.path.join(eddy_path,  f"{subjid}_eddy_corrected")
            eddy_out_DWIs=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii.gz")

            # ---------------------------
            # Create MASK
            # ---------------------------

            mask=os.path.join(eddy_path,f"{subjid}_unwarped")  #The mask is: sub-001_ses-01_
            in_B0s=os.path.join(topup_path, f"topup_out_unwarped_mean.nii")  #The mask esti. sub-200_ses-02_desc-brain_mask.nii takes the unwarped b0s, estiamate a mean and  use it to make a mask
            
            if not os.path.exists(mask):
                if do_mask == 10000000:
                    do_personalized_mask(in_B0s, mask, tt_log, method="bet", f=0.45, g=0)
                else:
                    subprocess.run(f"bet {in_B0s} {mask} -R -f 0.5 -g {g} -m", shell=True, check=True)

            # Since eddy current is one single line I'll run it directly here
            if not os.path.exists(eddy_out_DWIs):
                cmd = (
                    f"eddy "
                    f"--imain={in_DwIs} "
                    f"--mask={mask} "
                    f"--acqp={acqparams} "
                    f"--index={index} "
                    f"--bvecs={fslgrad}.bvec "
                    f"--bvals={fslgrad}.bval "
                    f"--topup={topup_out} "
                    f"--out={eddy_out} "
                    f"--flm=quadratic "
                    f"--slm=linear "
                    f"--dont_peas "
                    # f"--mb=2"
                    f"--residuals "
                    f"--verbose "
                    f"--estimate_move_by_susceptibility"
                )

                # --- Wait if we've hit the parallel job limit ---
                while len(processes) >= n_threads:
                    time.sleep(1)
                    processes = [p for p in processes if p.poll() is None]

                # --- Launch subprocess ---
                print(f"Active jobs: {len(processes)}/{n_threads}")
                parpool = subprocess.Popen(cmd, shell=True)
                processes.append(parpool)

    # --- Wait for ALL remaining jobs to finish ---
    for p in processes:
        p.wait()
    
    """
        Run FSL's Eddy-QUAD for automated quality assessment of eddy-corrected DWI data.

        Parameters
        ----------
        Compulsory arguments:
            eddyBase             Basename (including path) specified when running EDDY
        -idx, --eddyIdx      File containing indices for all volumes into acquisition parameters
        -par, --eddyParams   File containing acquisition parameters
        -m, --mask           Binary mask file
        -b, --bvals          b-values file

        Optional arguments:
            -g, --bvecs          b-vectors file - only used when <eddyBase>.eddy_residuals file is present
            -o, --output-dir     Output directory - default = '<eddyBase>.qc' 
            -f, --field          TOPUP estimated field (in Hz)
            -s, --slspec         Text file specifying slice/group acquisition
            -j, --json           JSON file specifying acquisition parameters (alternative for --slspec)
            -v, --verbose        Display debug messages

    """

Active jobs: 0/5

eddy diffusion --imain='my_ima' --acqp='my_acqp' ...


Reading images
Active jobs: 1/5

eddy diffusion --imain='my_ima' --acqp='my_acqp' ...


Reading images
Active jobs: 2/5

eddy diffusion --imain='my_ima' --acqp='my_acqp' ...


Reading images
Performing volume-to-volume registration
Running Register
Active jobs: 3/5

eddy diffusion --imain='my_ima' --acqp='my_acqp' ...


...................Allocated GPU # 0...................
Loading prediction maker
Evaluating prediction maker model
Calculating parameter updates
Iter: 0, Total mss = 34.3487
Loading prediction maker
Evaluating prediction maker model
Calculating parameter updates

Iter: 1, Total mss = 30.2016
Loading prediction maker
Evaluating prediction maker model
Calculating parameter updates
Reading images
Iter: 2, Total mss = 28.6728
Loading prediction maker
Evaluating prediction maker model
Calculating parameter updates
Iter: 3, Total mss = 28.0993
Loading prediction maker
Evaluating prediction maker model
Ca

## BIAS field correction

print("\n\n")
print("#################################\n")
print("##  BIAS field correction ...  ##\n")
print("#################################\n")
print("\n\n")

In [ ]:
if do_bias == 1: 
    NdoMinchiaSiamo="BIAS FIELD CORRECTION"
    """
    Perform B1 field inhomogeneity correction for a DWI volume series

    Options for ANTs N4BiasFieldCorrection command - I use the default

    -ants_b [100,3] N4BiasFieldCorrection option -b: [initial mesh resolution in mm, spline order] This value is optimised for human adult data and needs to be adjusted for rodent data.
    -ants_c [1000,0.0] N4BiasFieldCorrection option -c: [numberOfIterations,convergenceThreshold]
    -ants_s 4 N4BiasFieldCorrection option -s: shrink-factor applied to spatial dimensions
    
    ref. Tustison, N.; Avants, B.; Cook, P.; Zheng, Y.; Egan, A.; Yushkevich, P. & Gee, J. N4ITK: Improved N3 Bias Correction. IEEE Transactions on Medical Imaging, 2010, 29, 1310-1320 

    The script assumes that eddy current correction has already been performed
    
    """
    for vp in vps:
            id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
            project_id = f"sub-2{vp:02d}"  # project_id: project-specific code (sub-2XX) — yes, a bit redundant

            for session in sessions:    # Loop over sessions for this participant
                subjid = f"{id}_{session}"     # Standardized subject-session identifier 
                LOGsOut = os.path.join(script, "LOGs", f"{subjid}-LOGs")  # Log directory
                tt_log=os.path.join(script, "LOGs", f"{subjid}-LOGs",f"{subjid}-LOGs.txt" )
                
                dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
                eddy_path=os.path.join(derivatives, "dwi","eddy")
                biascorr=os.path.join(derivatives, "dwi","bias_corr")

                if not os.path.exists(biascorr):
                    os.makedirs(biascorr)
                 
                DWIin=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii.gz") # Eddy-corrected diffusion-weighted image (input for bias correction)
                BVECs=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.eddy_rotated_bvecs")  # Eddy-rotated b-vectors corresponding to the corrected DWIs
                BVALs=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bval")  # B-values file associated with the original acquisition
                bias_corrected=os.path.join(biascorr,  f"{subjid}_eddy_corrected_unbiased.nii.gz") # Output DWI after bias field correction
                bias=os.path.join(biascorr,  f"{subjid}_bias.nii.gz") # Estimated bias field image produced by ANTs
                residue_DwIs=os.path.join(biascorr,  f"{subjid}_residual.nii.gz") # Residual image computed as (eddy-corrected DWI − bias-corrected DWI)
                
                subprocess.run(f"dwibiascorrect ants {DWIin} {bias_corrected} -fslgrad {BVECs} {BVALs} -bias {bias}", shell=True)  # Apply N4 bias field correction using ANTs while preserving gradient information
                subprocess.run(f"fslmaths {DWIin} -sub {bias_corrected} {residue_DwIs}", shell=True)

                # ---------------------------
                # QC & visualization
                # ---------------------------
                if do_debug==1:
                    # Open supervised inspection on fsleyes for b0s, unwarped images, and estimated field
                    do_supervised_mode(DWIin, bias_corrected, residue_DwIs, bias)

                # Update log to record the Topup step
                LOGsOut_update(LOGsOut,NdoMinchiaSiamo,dwi_output=bias_corrected,sub_id=id, ses_id=session)


## SIGNAL drift correction

print("\n\n")
print("####################################\n")
print("##  BSIGNAL drift correction ...  ##\n")
print("####################################\n")
print("\n\n")

The signal drift correction has been implemented by Lisa K. 
It runs patch-wise 

In [ ]:
n_threads = 1
processes = []
if do_signaldrift == 1:
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)   
        raw_sub_path = os.path.join(prj_path, "derivatives", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
   
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")  # Log directory
            LOGs_file=os.path.join(script, "LOGs", f"{id}-LOGs",f"{subjid}-LOGs.txt" )
            derivatives = os.path.join(prj_path, "derivatives", id, session)   # Path to derivative folder for this participant & session

            ## Path setting
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            eddy_path=os.path.join(derivatives, "dwi","eddy")
            signal_drift=os.path.join(derivatives, "dwi","signaldrift")

            if not os.path.exists(signal_drift): 
                os.makedirs(signal_drift)

            DWIs_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii")
            BVECS_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.eddy_rotated_bvecs")
            BVALS_in=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bval") # add $set.bvec/bval
            OUTBASE_NAME=os.path.join(signal_drift,  f"{subjid}_eddy-corrected_signal-drift")


            # ---------------------------
            # Create MASK
            # ---------------------------

            mask=os.path.join(eddy_path,f"{subjid}_unwarped_mask.nii")  #The mask is: sub-001_ses-01_
    
            # Since eddy current is one single line I'll run it directly here
      
            cmd = (f"python3.12 signal_drift.py -i {DWIs_in} -bval {BVALS_in} -o {OUTBASE_NAME} -roi {mask}")

            # --- Wait if we've hit the parallel job limit ---
            while len(processes) >= n_threads:
                time.sleep(1)
                processes = [p for p in processes if p.poll() is None]

            # --- Launch subprocess ---
            print(f"Active jobs: {len(processes)}/{n_threads}")
            parpool = subprocess.Popen(cmd, shell=True)
            processes.append(parpool)
   
    # --- Wait for ALL remaining jobs to finish ---
    for p in processes:
        p.wait()

## INDEX estimation 
# This script can estimate following indices: 
-> DTI 
-> DKI 
-> Free water estimation
-> NODDI 

## Extra 
-> WMTI (Reconstruction of the diffusion signal with the WMTI model (DKI-MICRO)) 
-> MSDKI (Mean signal diffusion kurtosis imaging (MSDKI))

## DTI

In [12]:
if do_dtifit==1:  #Estimate dwi
    NdoMinchiaSiamo="DTIFIT"
    for vp in vps:
        id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
        raw_sub_path = os.path.join(prj_path, "derivatives", id)
        sessions = sorted([s for s in os.listdir(raw_sub_path)if s.startswith("ses-") and os.path.isdir(os.path.join(raw_sub_path, s))])
        
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            
            LOGsOut = os.path.join(script, "LOGs", f"{id}-LOGs")  # Log directory
            LOGs_file=os.path.join(script, "LOGs", f"{id}-LOGs",f"{subjid}-LOGs.txt" )
            derivatives = os.path.join(prj_path, "derivatives", id, session)   # Path to derivative folder for this participant & session

            ## Path setting
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            eddy_path=os.path.join(derivatives, "dwi","eddy")    

            Dwmri_index=os.path.join(derivatives, "dwi","index")
            Dtifit_out=os.path.join(Dwmri_index,"Tensor")
            dti_maps=os.path.join(Dtifit_out, str(subjid)+"_prep_signal-drift-corr")
            
            signal_drift=os.path.join(derivatives, "dwi","signaldrift")

            if not os.path.exists(signal_drift): 
                os.makedirs(signal_drift)

            DWIs_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii")
            BVECS_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.eddy_rotated_bvecs")
            BVALS_in=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bval") # add $set.bvec/bval
            OUTBASE_NAME=os.path.join(signal_drift,  f"{subjid}_eddy-corrected_signal-drift")

            if not os.path.exists(Dtifit_out): #Check if the folder exists
                os.makedirs(Dtifit_out)
            
            if not os.path.exists(Dwmri_index):
                os.makedirs(Dwmri_index)
            
            """
            Regarding the MASK the best option is to use a coregistered T1w2DWI mask, usually it has a better performance compared to BET or ANTsbrain ext.
            The coregistration is peformed using B0s and T1w, then the inverted matrix is applied to the MASK using a 0.4 thr
            """
            
            mask_IMG=os.path.join(eddy_path,f"{subjid}_unwarped_mask.nii.gz") 
            DWI_in=os.path.join(signal_drift,  f"{subjid}_eddy-corrected_signal-drift_corr.nii.gz")#os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii.gz")
            BVECS_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.eddy_rotated_bvecs")
            BVALS_in=os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.bval")

            """
            ## INDEX estiamtion
            # This script estimate following diffusion indeces 
            (0) dtifit 
            (1) Kurtosis 
            (2) Noddi 
            (3) Free water estimation diffusion tensor 
            
            It first set up all the folder strucutre, then create starts computing indices 
            the script also check if there are enough bshells 
            """

            print("\nLoading data, starting DTI estimation\n")

           # subprocess.run(f"dtifit -k {DWI_in} -o {dti_maps} -m {mask_IMG} -r {BVECS_in} -b {BVALS_in} -V ", shell=True)


            DWI_in=os.path.join(eddy_path,  f"{subjid}_eddy_corrected.nii.gz")
            dti_maps=os.path.join(Dtifit_out, str(subjid)+"_prep")
            subprocess.run(f"dtifit -k {DWI_in} -o {dti_maps} -m {mask_IMG} -r {BVECS_in} -b {BVALS_in} -V ", shell=True)


Loading data, starting DTI estimation

data file /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/eddy/sub-45_ses-01_eddy_corrected.nii.gz
mask file /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/eddy/sub-45_ses-01_unwarped_mask.nii.gz
bvecs     /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/eddy/sub-45_ses-01_eddy_corrected.eddy_rotated_bvecs
bvals     /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-45/ses-01/dwi/prep/sub-45_ses-01_dir-PA-AP_part-mag_dwi.bval
reading data
reading mask
ok
0 110 0 110 0 70
setting up vols
copying input properties to output volumes
zeroing output volumes
ok
Forming A matrix
starting the fits
0 slices processed
1 slices processed
2 slices processed
3 slices processed
4 slices processed
5 slices processed
6 slices processed
7 slices processed
8 slices processed
9 slices processed
10 slices processed
11 slices processed
12 slices processed
13 slices processed
14 slices processed
15 slic